In [ ]:

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
%run "./00_setup.ipynb"


In [ ]:

from betting.drawdown_controller import DrawdownController

ctrl = DrawdownController(peak_bankroll=100000)
print("## DrawdownController v5.4")
print(f"  peak_bankroll: {ctrl.peak_bankroll}")
print(f"  current_multiplier: {ctrl.get_multiplier(100000)}")
print(f"  recovery_state: {ctrl.get_state(100000).recovery_state}")
print("")
print("状態遷移: NORMAL → REDUCED → RECOVERING → NORMAL")
print("  REDUCED: DD悪化 + ROI低下で乗数を減少")
print("  RECOVERING: ROI回復で乗数を段階的に復帰")
print("  Nベット制限: 1ウィンドウ(20ベット)で最大+0.15まで")


In [ ]:
print("""
## ヒステリシス効果

REDUCED → RECOVERING の遷移条件:
  - ROI >= 0.98 が継続
  - 一定ベット数の良好な成績が必要

RECOVERING → NORMAL の遷移条件:
  - DD < 5% に回復

このヒステリシスにより:
  - 一時的な回復で即座に元の乗数に戻らない
  - 安定した回復が確認されてから復帰
  - REDUCED → RECOVERING → REDUCED のループを防止
""")

In [ ]:
print("""
## Nベット変更幅制限

max_adjustment_amount = 0.15

1ウィンドウ (20ベット) で乗数の変化量を +0.15 に制限:
  - RECOVERING: 1ベットあたり +0.05 → 20ベットで +1.0 だが cap で +0.15
  - ウィンドウ終了時にリセット → 次ウィンドウで再度 +0.15 可能

効果:
  - 急激なステーク増加を防止
  - 段階的な回復を保証
""")

In [ ]:
print("""
## 結論: DD×Rolling ROI 複合制御

DD制御による効果:
  1. 最大DD の縮小 (15-20% → 7-13%)
  2. ヒステリシスで安定的な回復
  3. Nベット制限で急激な変動を防止
  4. 全体の Sharpe ratio 向上

実データでの検証には BacktestEngine を使用。
""")